In [ ]:
# Interlude: Who Trains the Trainer? Learning by Experiment
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/interludes/learning-by-experiment.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = [
    {
        "path": "data/fashion-test.pt",
        "sha256": "1db79d080c51173c7df18a5e8389dd4ae20ecb0352a21be90aaa446aed612a09"
    },
    {
        "path": "data/fashion-train.pt",
        "sha256": "86a99167f14d98891de2bc34b83197727f89e178cdf9e5b019fceb7bf71cc427"
    }
]

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/interludes').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/interludes')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

assert _BOOK_ROOT.is_dir()

**Plan**

1. Implement the paired Fashion-MNIST experiment.
2. Define the reusable helpers: `TinyMLP`, `accuracy`, and `run_once`.

In [ ]:
from __future__ import annotations

import itertools
from copy import deepcopy

import matplotlib.pyplot as plt
import torch
from torch import Tensor, nn
import torch.nn.functional as F

# [1]
torch.set_num_threads(4)

development = torch.load("../../data/fashion-train.pt")
X_dev = development["X"].float().unsqueeze(1) / 255.0  # (1200, 1, 28, 28)
y_dev = development["y"]                                # (1200,)

split = torch.randperm(
    len(X_dev), generator=torch.Generator().manual_seed(6050)
)
fit_idx, val_idx = split[:1000], split[1000:]
X_fit, y_fit = X_dev[fit_idx], y_dev[fit_idx]
X_val, y_val = X_dev[val_idx], y_dev[val_idx]


# [2]
class TinyMLP(nn.Module):
    def __init__(self, use_batchnorm: bool) -> None:
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.bn1 = nn.BatchNorm1d(128) if use_batchnorm else nn.Identity()
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64) if use_batchnorm else nn.Identity()
        self.out = nn.Linear(64, 10)

    def forward(self, x: Tensor) -> Tensor:
        x = x.flatten(1)                            # (B, 1, 28, 28) -> (B, 784)
        x = F.relu(self.bn1(self.fc1(x)))           # (B, 784) -> (B, 128)
        x = F.relu(self.bn2(self.fc2(x)))           # (B, 128) -> (B, 64)
        return self.out(x)                          # (B, 64) -> (B, 10)


@torch.no_grad()
def accuracy(model: nn.Module, x: Tensor, target: Tensor) -> float:
    model.eval()                                    # freeze BatchNorm statistics
    return (model(x).argmax(1) == target).float().mean().item()


def run_once(
    seed: int,
    use_batchnorm: bool,
    learning_rate: float,
    endpoint: tuple[Tensor, Tensor] | None = None,
    epochs: int = 20,
) -> tuple[float, float]:
    torch.manual_seed(seed)
    model = TinyMLP(use_batchnorm)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=learning_rate, momentum=0.9
    )
    batch_rng = torch.Generator().manual_seed(seed + 100_000)
    best_validation = 0.0
    best_state = deepcopy(model.state_dict())

    for _ in range(epochs):
        model.train()
        order = torch.randperm(len(X_fit), generator=batch_rng)
        for idx in order.split(100):
            loss = F.cross_entropy(model(X_fit[idx]), y_fit[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        validation = accuracy(model, X_val, y_val)
        if validation > best_validation:
            best_validation = validation
            best_state = deepcopy(model.state_dict())

    model.load_state_dict(best_state)                # evaluate the selected checkpoint
    endpoint_score = (
        accuracy(model, *endpoint) if endpoint is not None else best_validation
    )
    return best_validation, endpoint_score

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Search four learning rates with paired seeds.
3. Report or visualize the measured result.

In [ ]:
# [1]
learning_rates = (0.003, 0.01, 0.03, 0.1)
search_seeds = tuple(range(6050, 6055))
search_rows: list[dict[str, float | bool | Tensor]] = []

# [2]
for use_batchnorm, learning_rate in itertools.product(
    (False, True), learning_rates
):
    scores = torch.tensor([
        run_once(seed, use_batchnorm, learning_rate)[0]
        for seed in search_seeds
    ])
    search_rows.append({
        "batchnorm": use_batchnorm,
        "learning_rate": learning_rate,
        "scores": scores,
        "mean": scores.mean().item(),
        "sd": scores.std(unbiased=True).item(),
    })

# [3]
for row in search_rows:
    label = "with BN" if row["batchnorm"] else "no BN  "
    print(
        f"{label}  lr={row['learning_rate']:<5}  "
        f"validation={100 * row['mean']:.1f}% +/- {100 * row['sd']:.1f}%"
    )

best_row = {
    use_batchnorm: max(
        (row for row in search_rows if row["batchnorm"] == use_batchnorm),
        key=lambda row: row["mean"],
    )
    for use_batchnorm in (False, True)
}

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Evaluate the locked configurations on the shared endpoint.

In [ ]:
# [1]
holdout = torch.load("../../data/fashion-test.pt")
X_test = holdout["X"].float().unsqueeze(1) / 255.0   # (600, 1, 28, 28)
y_test = holdout["y"]                                # (600,)
audit_seeds = tuple(range(6060, 6065))
audit: dict[bool, dict[str, float | Tensor]] = {}

# [2]
for use_batchnorm in (False, True):
    learning_rate = float(best_row[use_batchnorm]["learning_rate"])
    test_scores = torch.tensor([
        run_once(
            seed, use_batchnorm, learning_rate, endpoint=(X_test, y_test)
        )[1]
        for seed in audit_seeds
    ])
    audit[use_batchnorm] = {
        "learning_rate": learning_rate,
        "scores": test_scores,
        "mean": test_scores.mean().item(),
        "sd": test_scores.std(unbiased=True).item(),
    }
    label = "with BN" if use_batchnorm else "no BN  "
    print(
        f"{label}  locked lr={learning_rate:<4}  "
        f"test={100 * audit[use_batchnorm]['mean']:.1f}% "
        f"+/- {100 * audit[use_batchnorm]['sd']:.1f}%"
    )

scores_by_rate = {
    (bool(row["batchnorm"]), float(row["learning_rate"])): row["scores"]
    for row in search_rows
}
fixed_rate_contrasts = {
    learning_rate: (
        scores_by_rate[(True, learning_rate)]
        - scores_by_rate[(False, learning_rate)]
    )
    for learning_rate in learning_rates
}
paired_contrasts = {
    **{f"shared lr={rate:g}": values
       for rate, values in fixed_rate_contrasts.items()},
    "tuned validation": best_row[True]["scores"] - best_row[False]["scores"],
    "locked endpoint": audit[True]["scores"] - audit[False]["scores"],
}
print("paired BN - no-BN contrasts")
for label, contrasts in paired_contrasts.items():
    print(
        f"{label:<20} mean={100 * contrasts.mean():+.3f} pp  "
        f"SD={100 * contrasts.std(unbiased=True):.3f} pp"
    )